# Lab 1: ReAct 에이전트 기초

## ReAct = Reasoning + Acting
**생각하고(Thought) → 행동하고(Action) → 관찰하는(Observation) AI**

기존 LLM은 질문에 바로 답합니다. ReAct 에이전트는 다릅니다:

```
목표 입력
   ↓
Thought  : "먼저 센서 데이터를 확인해야 한다"
   ↓
Action   : check_sensor("line_A_bearing")
   ↓
Observation: "진동: 4.2mm/s ⚠️ 임계값 초과"
   ↓
Thought  : "이상 감지됨. 정비팀에 알림을 보내야 한다"
   ↓
... (목표 달성까지 반복)
   ↓
Final Answer
```

## 학습 목표

1. **ReAct 패턴** (Thought → Action → Observation 루프) 이해
2. LLM 없이 동작하는 **규칙 기반 ReAct 에이전트** 직접 구현
3. 제조 현장 도구 4개 정의 및 에이전트에 연결
4. 에이전트 실행 과정 **시각화**
5. **LangChain ReAct** 간단 소개

## 왜 ReAct가 제조 현장에서 중요한가?

| 기존 AI | ReAct 에이전트 |
|---------|----------------|
| 질문 → 즉시 답변 (블랙박스) | 생각 → 행동 → 관찰 (투명한 로그) |
| 외부 시스템 접근 불가 | 센서 DB, MES, ERP와 연동 |
| 단발성 답변 | 복합 목표 자율 달성 |
| 근거 없는 답변 (환각) | 도구 실행 결과 기반 판단 |

> **출처**: Yao et al. (2022). ReAct: Synergizing Reasoning and Acting in Language Models. ICLR 2023.

> ### 🎓 강사 가이드 — 1. 환경 설정
>
> | 항목 | 내용 |
> |------|------|
> | ⏱️ 소요 시간 | 약 5분 |
> | 🎯 핵심 목표 | ReAct Agent 실습 환경 확인 |
> | 🔑 핵심 개념 | LangChain, AgentExecutor, 도구 체계 |
>
> **💡 강조 포인트**
> - 이 실습은 LLM API 없이도 규칙 기반으로 ReAct 패턴을 완전히 구현합니다
> - 실제 서비스에서는 LLM API를 연결하면 됩니다


## 1. 환경 설정

In [1]:
# 기본 라이브러리만 사용 (외부 패키지 설치 불필요)
import datetime
import json
from typing import Dict, List, Callable, Optional, Any

print("✅ 기본 라이브러리 준비 완료")
print("   이 Lab은 LLM API 없이도 ReAct 패턴을 완전히 이해할 수 있습니다.")

✅ 기본 라이브러리 준비 완료
   이 Lab은 LLM API 없이도 ReAct 패턴을 완전히 이해할 수 있습니다.


> ### 🎓 강사 가이드 — 2. ReAct 핵심 컴포넌트 이해
>
> | 항목 | 내용 |
> |------|------|
> | ⏱️ 소요 시간 | 약 15분 |
> | 🎯 핵심 목표 | Thought → Action → Observation 사이클 이해 |
> | 🔑 핵심 개념 | ReAct = Reasoning + Acting, 루프 기반 추론 |
>
> **💡 강조 포인트**
> - Thought: LLM이 "무엇을 할지" 스스로 추론하는 단계
> - Action: 도구를 선택하고 실행하는 단계
> - Observation: 도구 실행 결과를 받아 다음 추론에 활용하는 단계
>
> **❓ 생각해보기**: "기존 IF-ELSE 규칙 시스템과 ReAct 에이전트의 가장 큰 차이는 무엇일까요?"


## 2. ReAct 에이전트 핵심 컴포넌트 이해

ReAct 에이전트를 구성하는 3가지 핵심 요소:

```
┌─────────────────────────────────────┐
│          ReAct 에이전트             │
│                                     │
│  ① Thought  : 현재 상황 분석 (LLM) │
│  ② Action   : 도구 선택 및 실행    │
│  ③ Observation: 실행 결과 확인      │
│                                     │
│  ┌────────────────────────────────┐│
│  │  Tool Registry (도구 창고)      ││
│  │  - check_sensor()              ││
│  │  - detect_anomaly()            ││
│  │  - send_alert()                ││
│  │  - create_work_order()         ││
│  └────────────────────────────────┘│
└─────────────────────────────────────┘
```

> ### 🎓 강사 가이드 — 3. 제조 현장 도구(Tool) 정의
>
> | 항목 | 내용 |
> |------|------|
> | ⏱️ 소요 시간 | 약 15분 |
> | 🎯 핵심 목표 | Tool 등록 방법과 LLM에 전달되는 도구 설명의 중요성 이해 |
> | 🔑 핵심 개념 | `@tool` 데코레이터, docstring = LLM에게 전달되는 도구 설명 |
>
> **💡 강조 포인트**
> - docstring이 명확할수록 LLM이 올바른 도구를 선택합니다
> - 입력/출력 형식을 docstring에 명시하면 파싱 에러가 줄어듭니다
>
> **🚨 주의사항**: 프롬프트 형식이 달라지면 Action 파싱 실패 — 형식을 엄격하게 유지하세요
>
> **❓ 생각해보기**: "에이전트가 잘못된 도구를 선택할 때 어떻게 수정할 수 있을까요?"


## 3. 제조 현장 도구(Tool) 정의

Tool은 에이전트가 현실 세계와 상호작용하는 창구입니다.  
실제 현장에서는 MES API, 센서 DB, ERP 시스템과 연결됩니다.

In [2]:
# ========================================
# 제조 현장 도구(Tool) 4개 정의
# ========================================

def check_sensor(sensor_id: str) -> str:
    """
    [도구 1] 설비 센서 값 조회
    
    실제 서비스에서는 MQTT 브로커나 센서 DB API를 호출합니다.
    현재는 시뮬레이션 데이터를 반환합니다.
    
    Args:
        sensor_id: 센서 ID (예: 'line_A_bearing', 'line_B_temp')
    
    Returns:
        센서 측정값 및 임계값 비교 결과
    """
    # 시뮬레이션 센서 데이터
    sensors = {
        "line_A_bearing":  {"value": 4.2, "unit": "mm/s",  "threshold": 3.0, "status": "초과"},
        "line_B_temp":     {"value": 68.0, "unit": "°C",   "threshold": 80.0, "status": "정상"},
        "line_C_pressure": {"value": 5.1, "unit": "bar",   "threshold": 6.0, "status": "정상"},
        "line_D_current":  {"value": 15.3, "unit": "A",    "threshold": 12.0, "status": "초과"},
        "line_E_vibration":{"value": 1.8, "unit": "mm/s",  "threshold": 3.0, "status": "정상"},
    }
    
    data = sensors.get(sensor_id)
    if not data:
        return f"ERROR: '{sensor_id}' 센서를 찾을 수 없습니다. 사용 가능: {list(sensors.keys())}"
    
    icon = "⚠️" if data["status"] == "초과" else "✅"
    return (
        f"{icon} {sensor_id}: {data['value']}{data['unit']} "
        f"(임계값: {data['threshold']}{data['unit']}, 상태: {data['status']})"
    )


def detect_anomaly(reading: str) -> str:
    """
    [도구 2] 센서 측정값의 이상 여부 판정
    
    실제 서비스에서는 ML 모델(AutoEncoder 등)을 호출합니다.
    
    Args:
        reading: 측정값 문자열 (예: 'vibration=4.2mm/s,threshold=3.0')
    
    Returns:
        이상 여부 판정 결과 및 심각도
    """
    try:
        # 입력 파싱: 'key=value' 형태 처리
        parts = {k: v for k, v in (item.split("=") for item in reading.split(","))}
        
        # 숫자만 추출 (단위 제거)
        import re
        value = float(re.sub(r'[^0-9.]', '', parts.get("vibration", parts.get("value", "0"))))
        threshold = float(re.sub(r'[^0-9.]', '', parts.get("threshold", "3.0")))
        
        ratio = value / threshold
        
        if ratio >= 1.5:
            severity = "위험"
            action = "즉시 가동 중단 및 점검 필요"
        elif ratio >= 1.0:
            severity = "경고"
            action = "4시간 이내 점검 권고"
        else:
            severity = "정상"
            action = "계속 모니터링"
        
        return f"판정: {severity} (측정={value}, 임계={threshold}, 비율={ratio:.2f}x) → {action}"
    except Exception as e:
        return f"ERROR: 이상 판정 실패 - 입력 형식 확인 (예: 'vibration=4.2,threshold=3.0') / {e}"


def send_alert(message: str, severity: str = "warning") -> str:
    """
    [도구 3] 정비팀 및 관리자에게 알림 전송
    
    실제 서비스에서는 SMS, Slack, 이메일 API를 호출합니다.
    
    Args:
        message: 알림 내용
        severity: 심각도 ('critical' | 'warning' | 'info')
    
    Returns:
        알림 전송 결과
    """
    # 심각도별 수신자 그룹 및 아이콘
    config = {
        "critical": {"icon": "🔴", "recipients": "정비팀장, 생산관리자, 공장장", "channel": "SMS+Slack"},
        "warning":  {"icon": "🟠", "recipients": "정비팀장, 생산관리자", "channel": "Slack"},
        "info":     {"icon": "🟡", "recipients": "정비팀", "channel": "Slack"},
    }
    cfg = config.get(severity, config["warning"])
    
    timestamp = datetime.datetime.now().strftime("%H:%M:%S")
    return (
        f"{cfg['icon']} [{severity.upper()}] {message} | "
        f"수신: {cfg['recipients']} | 채널: {cfg['channel']} | 시각: {timestamp}"
    )


def create_work_order(task: str, priority: str = "normal") -> str:
    """
    [도구 4] 정비 작업 지시서 생성 (MES 연동)
    
    실제 서비스에서는 MES(제조실행시스템) API를 호출합니다.
    
    Args:
        task: 작업 내용
        priority: 우선순위 ('urgent' | 'normal' | 'low')
    
    Returns:
        작업 지시서 ID 및 상세 정보
    """
    # 작업 지시서 ID 생성 (WO-날짜시간)
    order_id = f"WO-{datetime.datetime.now().strftime('%m%d-%H%M')}"
    
    priority_config = {
        "urgent": {"icon": "🔴", "hours": 2,  "dept": "정비1팀(긴급)"},
        "normal": {"icon": "🟡", "hours": 8,  "dept": "정비1팀"},
        "low":    {"icon": "🟢", "hours": 24, "dept": "정비2팀"},
    }
    cfg = priority_config.get(priority, priority_config["normal"])
    
    return (
        f"📋 작업 지시서 발행: {order_id} | 내용: {task} | "
        f"우선순위: {cfg['icon']} {priority} | "
        f"담당: {cfg['dept']} | 목표완료: {cfg['hours']}시간 이내"
    )


# 도구 목록
TOOLS = [check_sensor, detect_anomaly, send_alert, create_work_order]

print(f"✅ 제조 현장 도구 {len(TOOLS)}개 정의 완료")
for tool in TOOLS:
    first_line = tool.__doc__.strip().splitlines()[0]
    print(f"  - {tool.__name__}: {first_line}")

✅ 제조 현장 도구 4개 정의 완료
  - check_sensor: [도구 1] 설비 센서 값 조회
  - detect_anomaly: [도구 2] 센서 측정값의 이상 여부 판정
  - send_alert: [도구 3] 정비팀 및 관리자에게 알림 전송
  - create_work_order: [도구 4] 정비 작업 지시서 생성 (MES 연동)


> ### 🎓 강사 가이드 — 4. SimpleReActAgent 구현
>
> | 항목 | 내용 |
> |------|------|
> | ⏱️ 소요 시간 | 약 20분 |
> | 🎯 핵심 목표 | ReAct 루프를 직접 구현하며 내부 동작 원리 이해 |
> | 🔑 핵심 개념 | 루프 탈출 조건 (Final Answer), 최대 반복 횟수 설정 |
>
> **💡 강조 포인트**
> - 실제 LLM 없이도 규칙 기반으로 ReAct 패턴을 완전히 재현합니다
> - 루프가 무한히 돌지 않도록 `max_iterations`를 설정하는 것이 중요합니다
>
> **🚨 주의사항**: LLM이 "Final Answer:"를 출력하지 않으면 루프가 계속됩니다 — 탈출 조건 확인 필수


## 4. SimpleReActAgent 구현

LLM API 없이도 ReAct 패턴을 완전히 구현합니다.  
현재는 규칙 기반으로 Thought를 생성하지만, 실제 서비스에서는 LLM이 담당합니다.

In [3]:
class SimpleReActAgent:
    """
    규칙 기반 ReAct 에이전트 (LLM 없이 동작).
    
    실제 서비스에서는 think() 메서드를 LLM API 호출로 교체합니다.
    이 구현은 ReAct 패턴의 구조를 이해하기 위한 학습용입니다.
    """
    
    def __init__(self, tools: List[Callable]):
        """
        Args:
            tools: 에이전트가 사용할 도구 함수 목록
        """
        # 도구 이름 → 함수 매핑
        self.tools = {t.__name__: t for t in tools}
        
        # 실행 이력 (각 스텝의 Thought/Action/Observation 저장)
        self.history: List[Dict] = []
        
        # 사용 가능한 도구 목록 (LLM에게 제공하는 정보)
        self.tool_descriptions = {
            name: func.__doc__.strip().splitlines()[0]
            for name, func in self.tools.items()
        }
    
    def think(self, task: str, step: int, last_observation: Optional[str] = None) -> str:
        """
        [Thought 단계] 현재 상황을 분석하고 다음 행동을 결정합니다.
        
        실제 서비스에서는 LLM에게 다음을 전달:
        - 전체 목표(task)
        - 지금까지의 Thought/Action/Observation 이력
        - 사용 가능한 도구 목록
        그리고 LLM이 다음 Thought와 Action을 JSON으로 반환합니다.
        """
        # 규칙 기반 Thought 생성 (학습용 시뮬레이션)
        thoughts = {
            1: "작업을 시작한다. 먼저 해당 라인의 센서 데이터를 확인해야 한다.",
            2: f"센서 데이터를 확인했다. 이전 관측: '{last_observation[:50] if last_observation else '없음'}'. 이상 여부를 정밀 분석해야 한다.",
            3: f"이상 분석 결과를 받았다. 관련 팀에 즉시 알림을 발송해야 한다.",
            4: "알림을 발송했다. 공식 작업 지시서를 발행하여 정비를 체계적으로 진행해야 한다.",
            5: "작업 지시서까지 발행 완료. 모든 필요한 조치를 취했다. 작업을 종료한다.",
        }
        return thoughts.get(step, "상황을 종합적으로 판단 중...")
    
    def decide_action(self, task: str, step: int, last_observation: Optional[str] = None):
        """
        [Action 결정] 다음에 호출할 도구와 파라미터를 결정합니다.
        
        실제 서비스에서는 LLM이 JSON으로 출력:
        {"action": "check_sensor", "action_input": {"sensor_id": "line_A_bearing"}}
        """
        # 태스크에서 라인 정보 추출
        line = "A" if "A" in task else "B" if "B" in task else "A"
        
        # 단계별 행동 결정 (규칙 기반 시뮬레이션)
        actions = {
            1: ("check_sensor",    f"line_{line}_bearing"),
            2: ("detect_anomaly",  "vibration=4.2,threshold=3.0"),
            3: ("send_alert",      f"라인 {line} 베어링 이상 감지 - 즉시 점검 필요"),
            4: ("create_work_order", f"라인 {line} 베어링 교체 작업"),
            5: (None, None),  # 작업 완료
        }
        return actions.get(step, (None, None))
    
    def act(self, action_name: str, action_input: Any) -> str:
        """
        [Action 실행] 선택된 도구를 실행하고 결과를 반환합니다.
        """
        tool_fn = self.tools.get(action_name)
        if not tool_fn:
            return f"ERROR: 도구 '{action_name}' 없음. 사용 가능: {list(self.tools.keys())}"
        
        try:
            # 딕셔너리 입력이면 키워드 인자로, 문자열이면 위치 인자로 전달
            if isinstance(action_input, dict):
                return tool_fn(**action_input)
            else:
                return tool_fn(action_input)
        except Exception as e:
            return f"ERROR: {action_name} 실행 실패 - {str(e)}"
    
    def run(self, task: str, max_steps: int = 5) -> List[Dict]:
        """
        [ReAct 루프 실행] 목표 달성까지 Thought→Action→Observation 반복.
        
        Args:
            task: 에이전트에게 주어진 목표
            max_steps: 최대 실행 단계 수 (무한루프 방지)
        
        Returns:
            실행 이력 리스트
        """
        # 이력 초기화
        self.history = []
        last_observation = None
        
        print(f"🎯 목표: {task}")
        print(f"🔧 사용 가능한 도구: {list(self.tools.keys())}")
        print("=" * 60)
        
        for step in range(1, max_steps + 1):
            print(f"\n{'─' * 50}")
            print(f"  Step {step}/{max_steps}")
            print(f"{'─' * 50}")
            
            # ① Thought: 현재 상황 분석
            thought = self.think(task, step, last_observation)
            print(f"  💭 Thought : {thought}")
            
            # ② Action 결정
            action_name, action_input = self.decide_action(task, step, last_observation)
            
            # 종료 조건: action이 None이면 목표 달성
            if action_name is None:
                print(f"  ✅ Final Answer: 모든 조치 완료")
                break
            
            print(f"  ⚡ Action  : {action_name}({repr(action_input)})")
            
            # ③ 도구 실행 → Observation
            observation = self.act(action_name, action_input)
            print(f"  👁️  Observation: {observation}")
            
            # 이력 저장
            entry = {
                "step": step,
                "thought": thought,
                "action": action_name,
                "action_input": action_input,
                "observation": observation
            }
            self.history.append(entry)
            last_observation = observation
        
        print(f"\n{'=' * 60}")
        print(f"✅ 에이전트 실행 완료 (총 {len(self.history)}단계)")
        return self.history


print("✅ SimpleReActAgent 클래스 정의 완료")

✅ SimpleReActAgent 클래스 정의 완료


> ### 🎓 강사 가이드 — 5. 에이전트 실행: 시나리오 1 — 라인 A 베어링 이상 대응
>
> | 항목 | 내용 |
> |------|------|
> | ⏱️ 소요 시간 | 약 20분 |
> | 🎯 핵심 목표 | 실제 제조 시나리오에서 에이전트가 자율적으로 문제 해결하는 과정 관찰 |
> | 🔑 핵심 개념 | 에이전트 자율성, 다단계 추론, 도구 체이닝 |
>
> **💡 강조 포인트**
> - 에이전트가 어떤 순서로 도구를 호출하는지 Thought-Action-Observation을 따라가며 확인하세요
> - 사람이 했다면 어떤 순서로 점검했을지 비교해보세요
>
> **❓ 생각해보기**: "에이전트의 판단이 틀렸을 때 현장에서는 어떻게 개입해야 할까요?"


## 5. 에이전트 실행: 시나리오 1 - 라인 A 베어링 이상 대응

In [4]:
# 에이전트 인스턴스 생성
agent = SimpleReActAgent(tools=TOOLS)

# 시나리오 실행: 라인 A 베어링 이상 감지 → 조치
task = "라인 A 베어링 상태를 확인하고 이상이 있으면 알림 및 작업 지시서를 발행하라"

history = agent.run(task, max_steps=5)

🎯 목표: 라인 A 베어링 상태를 확인하고 이상이 있으면 알림 및 작업 지시서를 발행하라
🔧 사용 가능한 도구: ['check_sensor', 'detect_anomaly', 'send_alert', 'create_work_order']

──────────────────────────────────────────────────
  Step 1/5
──────────────────────────────────────────────────
  💭 Thought : 작업을 시작한다. 먼저 해당 라인의 센서 데이터를 확인해야 한다.
  ⚡ Action  : check_sensor('line_A_bearing')
  👁️  Observation: ⚠️ line_A_bearing: 4.2mm/s (임계값: 3.0mm/s, 상태: 초과)

──────────────────────────────────────────────────
  Step 2/5
──────────────────────────────────────────────────
  💭 Thought : 센서 데이터를 확인했다. 이전 관측: '⚠️ line_A_bearing: 4.2mm/s (임계값: 3.0mm/s, 상태: 초과)'. 이상 여부를 정밀 분석해야 한다.
  ⚡ Action  : detect_anomaly('vibration=4.2,threshold=3.0')
  👁️  Observation: 판정: 경고 (측정=4.2, 임계=3.0, 비율=1.40x) → 4시간 이내 점검 권고

──────────────────────────────────────────────────
  Step 3/5
──────────────────────────────────────────────────
  💭 Thought : 이상 분석 결과를 받았다. 관련 팀에 즉시 알림을 발송해야 한다.
  ⚡ Action  : send_alert('라인 A 베어링 이상 감지 - 즉시 점검 필요')
  👁️  Observation: 🟠

## 6. 실행 이력 요약 및 시각화

In [5]:
# 실행 이력을 테이블로 시각화
import pandas as pd

print("\n=== ReAct 실행 이력 요약 ===")

history_rows = []
for entry in history:
    history_rows.append({
        "단계": entry["step"],
        "Thought (요약)": entry["thought"][:50] + "...",
        "Action": entry["action"],
        "Observation (요약)": entry["observation"][:60] + "..."
    })

df = pd.DataFrame(history_rows)
print(df.to_string(index=False))

print(f"\n📊 통계:")
print(f"  총 실행 단계: {len(history)}")
print(f"  사용된 도구: {[h['action'] for h in history]}")
print(f"  성공한 단계: {sum(1 for h in history if 'ERROR' not in h['observation'])}")


=== ReAct 실행 이력 요약 ===
 단계                                          Thought (요약)            Action                                                Observation (요약)
  1               작업을 시작한다. 먼저 해당 라인의 센서 데이터를 확인해야 한다....      check_sensor            ⚠️ line_A_bearing: 4.2mm/s (임계값: 3.0mm/s, 상태: 초과)...
  2 센서 데이터를 확인했다. 이전 관측: '⚠️ line_A_bearing: 4.2mm/s (...    detect_anomaly             판정: 경고 (측정=4.2, 임계=3.0, 비율=1.40x) → 4시간 이내 점검 권고...
  3               이상 분석 결과를 받았다. 관련 팀에 즉시 알림을 발송해야 한다....        send_alert 🟠 [WARNING] 라인 A 베어링 이상 감지 - 즉시 점검 필요 | 수신: 정비팀장, 생산관리자 | 채널...
  4       알림을 발송했다. 공식 작업 지시서를 발행하여 정비를 체계적으로 진행해야 한다.... create_work_order 📋 작업 지시서 발행: WO-0315-1656 | 내용: 라인 A 베어링 교체 작업 | 우선순위: 🟡 nor...

📊 통계:
  총 실행 단계: 4
  사용된 도구: ['check_sensor', 'detect_anomaly', 'send_alert', 'create_work_order']
  성공한 단계: 4


## 7. 에이전트 실행: 시나리오 2 - 다른 라인 테스트

In [6]:
# 시나리오 2: 라인 B 온도 이상 확인 (정상 케이스)
# 도구 직접 호출로 빠른 테스트
print("=== 시나리오 2: 라인 B 온도 센서 직접 확인 ===")
print()

# 각 도구 개별 테스트
tests = [
    ("check_sensor",    "line_B_temp"),
    ("check_sensor",    "line_D_current"),
    ("detect_anomaly",  "vibration=1.8,threshold=3.0"),  # 정상 케이스
    ("detect_anomaly",  "vibration=5.5,threshold=3.0"),  # 위험 케이스
    ("send_alert",      "라인 D 전류 초과 감지"),
    ("create_work_order", "라인 D 전기 모터 점검"),
]

# 도구 함수 매핑
tool_map = {t.__name__: t for t in TOOLS}

for tool_name, tool_input in tests:
    result = tool_map[tool_name](tool_input)
    print(f"[{tool_name}] 입력: '{tool_input}'")
    print(f"  결과: {result}")
    print()

=== 시나리오 2: 라인 B 온도 센서 직접 확인 ===

[check_sensor] 입력: 'line_B_temp'
  결과: ✅ line_B_temp: 68.0°C (임계값: 80.0°C, 상태: 정상)

[check_sensor] 입력: 'line_D_current'
  결과: ⚠️ line_D_current: 15.3A (임계값: 12.0A, 상태: 초과)

[detect_anomaly] 입력: 'vibration=1.8,threshold=3.0'
  결과: 판정: 정상 (측정=1.8, 임계=3.0, 비율=0.60x) → 계속 모니터링

[detect_anomaly] 입력: 'vibration=5.5,threshold=3.0'
  결과: 판정: 위험 (측정=5.5, 임계=3.0, 비율=1.83x) → 즉시 가동 중단 및 점검 필요

[send_alert] 입력: '라인 D 전류 초과 감지'
  결과: 🟠 [WARNING] 라인 D 전류 초과 감지 | 수신: 정비팀장, 생산관리자 | 채널: Slack | 시각: 16:56:13

[create_work_order] 입력: '라인 D 전기 모터 점검'
  결과: 📋 작업 지시서 발행: WO-0315-1656 | 내용: 라인 D 전기 모터 점검 | 우선순위: 🟡 normal | 담당: 정비1팀 | 목표완료: 8시간 이내



> ### 🎓 강사 가이드 — 8. LangChain ReAct 소개
>
> | 항목 | 내용 |
> |------|------|
> | ⏱️ 소요 시간 | 약 15분 |
> | 🎯 핵심 목표 | 규칙 기반 → 실제 LLM 기반 ReAct로 전환하는 방법 이해 |
> | 🔑 핵심 개념 | LangChain AgentExecutor, LLM + Tools 통합 |
>
> **💡 강조 포인트**
> - 앞서 직접 구현한 ReAct 루프를 LangChain이 대신 처리해줍니다
> - `AgentExecutor`는 도구 선택, 실행, 결과 파싱을 자동화합니다
>
> **❓ 생각해보기**: "LLM API 비용이 걱정된다면 어떤 전략으로 에이전트를 배포해야 할까요?"


## 8. LangChain ReAct 소개

실제 서비스에서는 LangChain `AgentExecutor`를 사용합니다.  
LLM이 자동으로 Thought를 생성하고 도구를 선택합니다.

In [7]:
# LangChain ReAct 코드 구조 미리보기
# (패키지 설치 및 API 키 없이 코드 구조만 학습)

langchain_react_preview = '''
# LangChain으로 ReAct 에이전트 구현 (실제 서비스용)
# pip install langchain langchain-openai

from langchain.agents import create_react_agent, AgentExecutor
from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langchain import hub

# ① 도구 정의 (@tool 데코레이터 사용)
@tool
def check_sensor_tool(sensor_id: str) -> str:
    """
    설비 센서 데이터를 조회합니다.
    Args:
        sensor_id: 센서 ID (예: \'line_A_bearing\')
    Returns:
        센서 측정값 및 임계값 비교 결과
    """
    return check_sensor(sensor_id)  # 위에서 정의한 함수 재사용

@tool
def send_alert_tool(message: str) -> str:
    """정비팀에 이상 알림을 전송합니다."""
    return send_alert(message)

tools = [check_sensor_tool, send_alert_tool]

# ② LLM 설정 (GPT-4.1-mini 사용)
llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0,  # 일관된 답변을 위해 temperature=0
    api_key="YOUR_OPENAI_API_KEY"
)

# ③ ReAct 프롬프트 (LangChain Hub에서 공식 프롬프트 사용)
prompt = hub.pull("hwchase17/react")

# ④ 에이전트 생성
agent = create_react_agent(llm=llm, tools=tools, prompt=prompt)

# ⑤ AgentExecutor로 실행
executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,        # Thought/Action/Observation 출력
    max_iterations=5,    # 최대 5번 반복 (무한루프 방지)
    handle_parsing_errors=True
)

# ⑥ 실행
result = executor.invoke({
    "input": "라인 A 베어링 센서를 확인하고 이상이 있으면 알림을 보내세요"
})
print(result["output"])

# LangSmith에서 자동으로 전체 실행 과정이 추적됩니다!
# → https://smith.langchain.com 에서 Trace 확인
'''

print("=== LangChain ReAct 에이전트 코드 구조 ===")
print(langchain_react_preview)

print("\n💡 현재 Lab에서 구현한 SimpleReActAgent와 LangChain ReAct의 차이점:")

comparison = pd.DataFrame({
    "항목": ["Thought 생성", "Action 결정", "도구 호출", "LangSmith 연동", "프로덕션 적합"],
    "SimpleReActAgent (이 Lab)": ["규칙 기반 (하드코딩)", "규칙 기반 (하드코딩)", "동일", "❌", "학습용"],
    "LangChain ReAct (실서비스)": ["LLM이 자동 생성", "LLM이 자동 결정 (JSON)", "동일", "✅ 자동", "✅"]
})
print(comparison.to_string(index=False))

=== LangChain ReAct 에이전트 코드 구조 ===

# LangChain으로 ReAct 에이전트 구현 (실제 서비스용)
# pip install langchain langchain-openai

from langchain.agents import create_react_agent, AgentExecutor
from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langchain import hub

# ① 도구 정의 (@tool 데코레이터 사용)
@tool
def check_sensor_tool(sensor_id: str) -> str:
    """
    설비 센서 데이터를 조회합니다.
    Args:
        sensor_id: 센서 ID (예: 'line_A_bearing')
    Returns:
        센서 측정값 및 임계값 비교 결과
    """
    return check_sensor(sensor_id)  # 위에서 정의한 함수 재사용

@tool
def send_alert_tool(message: str) -> str:
    """정비팀에 이상 알림을 전송합니다."""
    return send_alert(message)

tools = [check_sensor_tool, send_alert_tool]

# ② LLM 설정 (GPT-4.1-mini 사용)
llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0,  # 일관된 답변을 위해 temperature=0
    api_key="YOUR_OPENAI_API_KEY"
)

# ③ ReAct 프롬프트 (LangChain Hub에서 공식 프롬프트 사용)
prompt = hub.pull("hwchase17/react")

# ④ 에이전트 생성
agent = create_react_agent(llm=llm, tools=tools

In [8]:
# LangSmith 연동 설정 방법 안내
import os

print("=== LangSmith 연동 설정 방법 ===")
print()
print("1. https://smith.langchain.com 에서 계정 생성")
print("2. Settings → API Keys → Create API Key")
print("3. .env 파일 또는 환경변수 설정:")
print()
print("   LANGCHAIN_TRACING_V2=true")
print("   LANGCHAIN_API_KEY=ls-xxxxxxxx")
print("   LANGCHAIN_PROJECT=manufacturing-ai-koreatech")
print()
print("4. 기존 코드 변경 없이 자동으로 모든 단계가 추적됩니다!")
print()

# 환경변수 설정 코드 (실제 키 있을 때 사용)
# os.environ["LANGCHAIN_TRACING_V2"] = "true"
# os.environ["LANGCHAIN_API_KEY"] = "ls-your-key-here"
# os.environ["LANGCHAIN_PROJECT"] = "manufacturing-ai-koreatech"

print("✅ 현재: Mock 모드 실행 중 (API 키 없이도 전체 학습 가능)")
print()
print("LangSmith에서 확인할 수 있는 정보:")
print("  - 각 단계(Thought/Action/Observation) 소요 시간")
print("  - 토큰 사용량 및 비용")
print("  - 실패한 단계 및 원인 분석")
print("  - 에이전트 성능 비교 (A/B 테스트)")

=== LangSmith 연동 설정 방법 ===

1. https://smith.langchain.com 에서 계정 생성
2. Settings → API Keys → Create API Key
3. .env 파일 또는 환경변수 설정:

   LANGCHAIN_TRACING_V2=true
   LANGCHAIN_API_KEY=ls-xxxxxxxx
   LANGCHAIN_PROJECT=manufacturing-ai-koreatech

4. 기존 코드 변경 없이 자동으로 모든 단계가 추적됩니다!

✅ 현재: Mock 모드 실행 중 (API 키 없이도 전체 학습 가능)

LangSmith에서 확인할 수 있는 정보:
  - 각 단계(Thought/Action/Observation) 소요 시간
  - 토큰 사용량 및 비용
  - 실패한 단계 및 원인 분석
  - 에이전트 성능 비교 (A/B 테스트)


## 9. 핵심 정리

### ReAct 패턴 정리

```
사용자 목표 입력
       ↓
┌─── ReAct 루프 ────────────────────────────────┐
│                                               │
│  💭 Thought: 현재 상황 분석 + 다음 행동 계획   │
│       ↓                                      │
│  ⚡ Action: 도구 선택 및 실행                  │
│       ↓                                      │
│  👁️ Observation: 도구 실행 결과 수신           │
│       ↓                                      │
│  [목표 달성?] → 아니오 → 루프 반복             │
│                → 예 → Final Answer            │
└───────────────────────────────────────────────┘
```

### 제조 현장 ReAct 에이전트 활용 예시

| 시나리오 | 사용 도구 | 결과물 |
|---------|----------|--------|
| 설비 이상 자동 대응 | 센서 조회 → 이상 판정 → 알림 → 지시서 | 작업 지시서 |
| 품질 불량 원인 분석 | 품질 조회 → 이력 검색 → 보고서 생성 | 원인 분석 보고서 |
| 부품 재고 부족 처리 | 재고 확인 → 발주 → 일정 조정 | 발주서 |

### 다음 Lab (02_manufacturing_tools.ipynb)
- 제조 현장 특화 도구 4개 심화 구현
- 멀티-턴 대화(이전 대화 기억) 구현
- 에이전트 한계 및 Production 배포 고려사항

---

## 📝 과제 — S4 Lab 1: ReAct 에이전트 기초

### 기본 과제 (필수)
1. **시나리오 실행 분석**: "현재 라인 A의 베어링 이상 상태를 확인하고 조치 권고사항을 알려줘" 에이전트 실행 결과를 캡처하고, Thought-Action-Observation 3 사이클을 표로 정리하세요

   | 사이클 | Thought (에이전트 추론) | Action (호출한 도구) | Observation (결과) |
   |--------|------------------------|---------------------|---------------------|
   | 1 | | | |
   | 2 | | | |
   | 3 | | | |

2. **에이전트 한계 찾기**: 에이전트가 잘못 판단하거나 답변하지 못한 케이스를 찾아 원인을 분석하세요

### 심화 과제 (선택)
1. **새 Tool 추가**: "생산 실적 조회" Tool을 새로 설계하고 구현하세요. 에이전트가 이 Tool을 활용하는 시나리오를 만들어 실행하세요
   - 힌트: Tool은 설비 ID를 입력받아 오늘의 생산량, 불량률, 가동률을 반환해야 합니다
2. **오류 복구 로직**: Tool 실행이 실패했을 때(예: 설비 연결 끊김) 에이전트가 자동으로 재시도하거나 대안을 찾는 로직을 구현하세요

### 제출 기준
- [ ] 모든 셀 오류 없이 실행 완료
- [ ] Thought-Action-Observation 3 사이클 표 작성
- [ ] 에이전트 한계 케이스 분석 포함
- [ ] (심화) 새 Tool 구현 또는 오류 복구 로직 코드 포함
